# GP1 ASR — Kaggle Submission Notebook

**Purpose:** Inference-only submission template.
Weights are downloaded from a GitHub release; no training happens here.

**Submission flow:**
1. Set `RELEASE_TAG` in the configuration cell below.
2. Run all cells top-to-bottom.
3. Submit `/kaggle/working/submission.csv`.

> Source repo: https://github.com/DKazhekin/ITMO_Speech_Recognition_Course


In [ ]:
# Install extra dependencies not pre-installed in the Kaggle environment.
# torch / torchaudio are already present; we only add NLP + audio extras.
!pip install -q num2words pyctcdecode kenlm pyyaml torchaudio soundfile


## 1. Configure release

In [ ]:
# ---- Edit these values before running ----
RELEASE_OWNER = "DKazhekin"
RELEASE_REPO  = "ITMO_Speech_Recognition_Course"
RELEASE_TAG   = "v0.1.0"   # <-- set to the published release tag
BASELINE      = "quartznet"

# Kaggle competition data mount point
KAGGLE_DATA_ROOT = "/kaggle/input/asr-2026-spoken-numbers-recognition-challenge"


## 2. Download release artifacts

In [ ]:
import urllib.request
from pathlib import Path

RELEASE_DIR = Path("/kaggle/working/release")
RELEASE_DIR.mkdir(parents=True, exist_ok=True)

_BASE_URL = (
    f"https://github.com/{RELEASE_OWNER}/{RELEASE_REPO}"
    f"/releases/download/{RELEASE_TAG}"
)

def _download_asset(name: str, required: bool = True) -> Path:
    """Download a single release asset from GitHub Releases.

    Docs: https://cli.github.com/manual/gh_release_download
    """
    url = f"{_BASE_URL}/{name}"
    dest = RELEASE_DIR / name
    try:
        print(f"Downloading {url} ...")
        urllib.request.urlretrieve(url, dest)
        print(f"  -> saved to {dest} ({dest.stat().st_size:,} bytes)")
    except Exception as exc:
        if required:
            raise RuntimeError(f"Failed to download required asset '{name}': {exc}") from exc
        print(f"  -> optional asset '{name}' not available ({exc}); skipping.")
    return dest

_download_asset("model.pt",     required=True)
_download_asset("config.yaml",  required=True)
_download_asset("release.json", required=True)
_download_asset("lm.bin",       required=False)   # optional KenLM binary


## 3. Clone gp1 source

In [ ]:
import subprocess
import sys

REPO_DIR = "/kaggle/working/repo"
GP1_SRC  = f"{REPO_DIR}/group-projects/gp1/src"

subprocess.run(
    [
        "git", "clone",
        "--depth", "1",
        "--branch", RELEASE_TAG,
        f"https://github.com/{RELEASE_OWNER}/{RELEASE_REPO}.git",
        REPO_DIR,
    ],
    check=True,
)
if GP1_SRC not in sys.path:
    sys.path.insert(0, GP1_SRC)
print(f"gp1 source added to sys.path: {GP1_SRC}")


## 4. Build test manifest from Kaggle input

In [ ]:
from pathlib import Path
from gp1.data.manifest import build_manifest, read_jsonl

TEST_CSV        = Path(KAGGLE_DATA_ROOT) / "test.csv"
TEST_AUDIO_ROOT = Path(KAGGLE_DATA_ROOT)
MANIFEST_PATH   = Path("/kaggle/working/test_manifest.jsonl")

n_records = build_manifest(TEST_CSV, TEST_AUDIO_ROOT, MANIFEST_PATH)
print(f"Built manifest with {n_records} records -> {MANIFEST_PATH}")

records = read_jsonl(MANIFEST_PATH)
print(f"Loaded {len(records)} ManifestRecord objects.")


## 5. Run inference → submission.csv

In [ ]:
import csv
import torch
from pathlib import Path
from gp1.submit.inference import InferenceConfig, run_inference

_lm_path = Path("/kaggle/working/release/lm.bin")

cfg = InferenceConfig(
    checkpoint_path=Path("/kaggle/working/release/model.pt"),
    config_path=Path("/kaggle/working/release/config.yaml"),
    lm_binary_path=_lm_path if _lm_path.exists() else None,
    batch_size=32,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

print(f"Device: {cfg.device}")
print(f"LM enabled: {cfg.lm_binary_path is not None}")

preds = run_inference(records, cfg)

_submission_path = "/kaggle/working/submission.csv"
with open(_submission_path, "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["filename", "transcription"])
    for fn, digits in preds:
        w.writerow([fn, digits])

print(f"Wrote {len(preds)} predictions to {_submission_path}")


## 6. Preview submission

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/working/submission.csv")
print(f"Submission shape: {df.shape}")
df.head()
